In [1]:
import pandas as pd

# Load your dataset
df = pd.read_csv(r"C:\Users\dell\Downloads\archive\student_performance.csv")

print(df.head())
print(df.info())

   StudyHours  Attendance  Resources  Extracurricular  Motivation  Internet  \
0          19          64          1                0           0         1   
1          19          64          1                0           0         1   
2          19          64          1                0           0         1   
3          19          64          1                1           0         1   
4          19          64          1                1           0         1   

   Gender  Age  LearningStyle  OnlineCourses  Discussions  \
0       0   19              2              8            1   
1       0   23              3             16            0   
2       0   28              1             19            0   
3       0   19              2              8            1   
4       0   23              3             16            0   

   AssignmentCompletion  ExamScore  EduTech  StressLevel  FinalGrade  
0                    59         40        0            1           3  
1               

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le_dict = {}

for col in df.columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        le_dict[col] = le

# Define target (CHANGE based on your dataset)
target = "FinalGrade"   # or "Exam_Score" or "Performance"

X = df.drop(columns=["FinalGrade", "ExamScore"])
y = df[target]
print(y.value_counts())
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

FinalGrade
0    3832
2    3618
1    3310
3    3243
Name: count, dtype: int64


In [3]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Random Forest Results:")
evaluate_model(rf, X_test, y_test)

Random Forest Results:
Accuracy: 0.9239557300963941
F1 Score: 0.9239049034730297
Precision: 0.9241290570278333
Recall: 0.9239557300963941


In [7]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss'
)

xgb.fit(X_train, y_train)

print("\nXGBoost Results:")
evaluate_model(xgb, X_test, y_test)

scores = cross_val_score(xgb, X, y, cv=5, scoring='accuracy')
print("CV Accuracy:", scores.mean())


XGBoost Results:
Accuracy: 0.8457693680828275
F1 Score: 0.8454973118900484
Precision: 0.8465811162184279
Recall: 0.8457693680828275
CV Accuracy: 0.24316358953434997


In [9]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(verbose=0)
cat.fit(X_train, y_train)

print("\nCatBoost Results:")
evaluate_model(cat, X_test, y_test)


CatBoost Results:
Accuracy: 0.8504105676544091
F1 Score: 0.8502901128515306
Precision: 0.8506180183384473
Recall: 0.8504105676544091


In [10]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC()
svm.fit(X_train_scaled, y_train)

print("\nSVM Results:")
evaluate_model(svm, X_test_scaled, y_test)


SVM Results:
Accuracy: 0.3802213495180293
F1 Score: 0.3754489922133472
Precision: 0.38173784944320377
Recall: 0.3802213495180293


In [14]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

print("\nLightGBM Results:")
evaluate_model(lgbm, X_test, y_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000879 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 188
[LightGBM] [Info] Number of data points in the train set: 11202, number of used features: 14
[LightGBM] [Info] Start training from score -1.295719
[LightGBM] [Info] Start training from score -1.442288
[LightGBM] [Info] Start training from score -1.353453
[LightGBM] [Info] Start training from score -1.462891

LightGBM Results:
Accuracy: 0.8014994644769725
F1 Score: 0.8009868254497641
Precision: 0.8026513361366404
Recall: 0.8014994644769725


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "RandomForest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "CatBoost": cat,
    "SVM":svm
}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f"{name} CV Accuracy: {scores.mean():.4f}")

RandomForest CV Accuracy: 0.9135
XGBoost CV Accuracy: 0.8430
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000638 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 188
[LightGBM] [Info] Number of data points in the train set: 11202, number of used features: 14
[LightGBM] [Info] Start training from score -1.295719
[LightGBM] [Info] Start training from score -1.442288
[LightGBM] [Info] Start training from score -1.353453
[LightGBM] [Info] Start training from score -1.462891
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000633 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 188
[LightGBM] [Info] Number of data points in the train set: 11202, number of used features: 14
[LightGBM] [Info] S

KeyboardInterrupt: 

In [16]:
from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('cat', cat),
        ('xgb', xgb)
    ],
    voting='soft'
)

voting.fit(X_train, y_train)

evaluate_model(voting, X_test, y_test)

Accuracy: 0.9061049625133881
F1 Score: 0.9059806320933218
Precision: 0.9061290134223859
Recall: 0.9061049625133881


In [17]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack = StackingClassifier(
    estimators=[
        ('rf', rf),
        ('cat', cat),
        ('xgb', xgb)
    ],
    final_estimator=LogisticRegression()
)

stack.fit(X_train, y_train)

evaluate_model(stack, X_test, y_test)

Accuracy: 0.9225276686897537
F1 Score: 0.9225217187095531
Precision: 0.922703839315217
Recall: 0.9225276686897537
